# S03 · 分离调试与完整训练的验收标准

本课生成可审查的实验计划，并核对官方调试参数。它不输出一个未经验证的“一键训练成功”命令。完整 Megatron+SGLang 作业必须结合模型转换、硬件、并行划分与容器版本确定。

## 1 · 先定位生成侧还是训练侧

官方支持 `--debug-rollout-only`、`--debug-train-only`，以及保存/重放 rollout 数据。使用固定 rollout 可以减少采样变化，帮助比较训练实现；只完成 rollout 不代表优化器和权重同步正确。

保存/加载路径支持 `{rollout_id}` 格式。读取 debug 的 PyTorch 文件仅限自己生成的可信产物。


In [ ]:
import json
import course
root=course.source()
doc=(root/'docs/en/developer_guide/debug.md').read_text(encoding='utf-8')
flags=['--debug-rollout-only','--debug-train-only','--save-debug-rollout-data','--load-debug-rollout-data','--save-debug-train-data']
for flag in flags:
    assert flag in doc
    print(flag)
print('PASS: 固定版本官方文档包含这些参数；未执行 CLI 解析或训练。')


## 2 · 三阶段调试计划

| 阶段 | 必须观察 | 不足以证明什么 |
|---|---|---|
| rollout-only | 模板、token、EOS、奖励、失败率 | 反向传播正确 |
| 固定数据 train-only | mask、log-prob、优势、梯度、参数变化 | 训练权重已发给生成服务 |
| 完整至少两轮 | 参数同步、后续样本版本、保存和恢复 | 稳定泛化或多种子收益 |

至少两轮是为了检查训练更新后的生成。第一次 rollout 可能仍是初始权重；只看第一轮 reward 无法证明同步生效。若使用异步采样，需要同时记录权重版本、样本滞后与 off-policy 修正规则。

不要把 actor log-prob、rollout log-prob、reference log-prob 视为同一对象。采样温度、top-p、tokenization、数值精度与权重版本都可能造成差异，先核对定义再判断数值是否应该相等。


In [ ]:
plan=dict(source_revision=course.REVISION,
    hardware={'current_reference':'RTX 4060 Laptop 8GB','full_training_verified':False},
    environment={'image_digest':None,'driver':None,'sglang_version':None,'megatron_revision':None},
    task={'name':'strict_yes_no','reward_entry':'course.strict_reward','batch_reward_entry':'course.batch_reward',
          'data_split_hash':None,'model_revision':None},
    stages=['rollout-only','fixed-rollout train-only','two-or-more full iterations','checkpoint resume'],
    acceptance=['No silent token/mask mismatch','Report generation errors separately',
                'Verify actor parameter change','Verify synchronized generation weights',
                'Recover optimizer and data progress','Evaluate held-out prompts'])
course.save_report('03_run_plan.json',plan)
print(json.dumps(plan,indent=2))
missing=[key for key,value in plan['environment'].items() if value is None]
assert missing
print('尚未填写的环境条件:',missing,'；该计划不可当作已验证启动配置。')


## 3 · 参数迁移与资源核对

Megatron 参数直接进入其训练配置；SGLang 参数通常增加 `--sglang-` 前缀，例如内存参数 `--sglang-mem-fraction-static`。这不是说任意新版引擎参数都能在旧镜像里使用，必须检查实际版本的参数解析。

总 batch 需要区分提示数、每提示采样数、序列数、micro batch、数据并行与累积。Tensor/Pipeline/Context Parallel 改变模型与序列的切分，不能简单都乘成训练样本数。共享 GPU 的训练/生成切换，还要考虑权重、KV cache、offload 和主机 RAM。

课程不建议在 8GB 设备直接照搬数据中心卡的模型和配方。更小模型或 offload 能否成功，需要独立实测，不能仅由参数量推断。

## 4 · 本系列下一步

在独立且兼容的 GPU 环境确定后，依次完成官方最小配方、真实自定义奖励、多轮工具轨迹、异步采样与恢复实验。每一节都保留配置、失败日志、逐样本输出和验收报告。

验收作业：给出你准备运行的镜像 digest、源码 commit、模型 revision、硬件拓扑和数据哈希；说明出错时如何切到固定 rollout 重放。仅打印配置或创建 Sample 不计作完成真实训练。
